In [1]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TextClassificationPipeline
import torch


### Load tained model

In [2]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-cased')

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at b

In [3]:
model.load_state_dict(torch.load('state.pt'))

<All keys matched successfully>

In [4]:
model = model.eval()

### load and Tokenize MRPC test

In [5]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

dataset_test = load_dataset('glue', 'mrpc', split='test')

#Encode dataset
def encode(examples):
    return tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding='max_length')

dataset_test = dataset_test.map(encode, batched=True)
#Rename label to labels
dataset_test = dataset_test.map(lambda rows: {'labels': rows['label']}, batched=True)

#Set dataset format and dataloader
dataset_test.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=16)

Reusing dataset glue (C:\Users\sago\.cache\huggingface\datasets\glue\mrpc\1.0.0\dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
Parameter 'function'=<function encode at 0x00000256800913A0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


  0%|          | 0/2 [00:00<?, ?ba/s]

  0%|          | 0/2 [00:00<?, ?ba/s]

In [6]:
results = []
for i, batch in enumerate(test_loader):
    _, outputs = model(**batch)
    outputs = torch.max(outputs, 1).indices.numpy()
    results.extend(outputs)


TypeError: max() received an invalid combination of arguments - got (str, int), but expected one of:
 * (Tensor input)
 * (Tensor input, Tensor other, *, Tensor out)
 * (Tensor input, int dim, bool keepdim, *, tuple of Tensors out)
 * (Tensor input, name dim, bool keepdim, *, tuple of Tensors out)


In [9]:
# Calculate accuracy
correct_predictions = 0
total_predictions = len(dataset_test)
for predicted, example in zip(results, dataset_test):
    predicted_label = predicted
    ground_truth_label = example['labels']
    if predicted_label == ground_truth_label:
        correct_predictions += 1

accuracy = correct_predictions / total_predictions

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 80.17%
